In [6]:
pip install kaggle

In [7]:
# configure the path of the kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


importing the dataset
!kaggle datasets download -d USERNAME/DATASET-NAME


In [8]:
# API TO FETCH THE DATASET FROM KAGGLE
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:03<00:00, 22.0MB/s]



In [9]:
# extracting the compressed dataset

from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('The dataset is extracted')

The dataset is extracted


importing librairies



In [10]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import  PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [11]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**Since you're working with sentiment analysis, stop-word removal is one of the common text preprocessing steps.**

In [12]:
# printing the stopwords in  english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

DATA PROCESSING


In [13]:
#loading the data from csv file to pandas dataframe
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',encoding='ISO-8859-1')

In [14]:
twitter_data.shape

(1599999, 6)

In [15]:
 twitter_data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [16]:
# naming the columns and reading the dataset again
column_names = ['target','id','date','flag','user','text']
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',encoding='ISO-8859-1',names=column_names)

In [17]:
twitter_data.shape

(1600000, 6)

In [18]:
twitter_data.columns

Index(['target', 'id', 'date', 'flag', 'user', 'text'], dtype='object')

In [19]:
twitter_data.head()

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [20]:
# counting the number of missing values in the dataset
twitter_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


In [21]:
#checking the distribution of target column
twitter_data['target'].value_counts()

,count
target,
0,800000
4,800000


convert the target "4" to "1"

In [22]:
twitter_data.replace({'target':{4:1}}, inplace=True)

In [23]:
#checking the distribution of target column
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


0 -- > Negative tweet
1 -- > Positive tweet

**STEMMING**

Stemming is the process of reducing a word to its Root word



---



example : actor, actress, acting = act

In [24]:
port_stem = PorterStemmer()

In [25]:
def stemming(content):

  stemmed_content = re.sub(r'[^a-zA-Z]', ' ', content)  # remove non letter
  stemmed_content = stemmed_content.lower() # convert everything to lowercase
  stemmed_content = stemmed_content.split() # split the sentence into words like  ["i", "love", "this", "movie"]
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')] # stemming
  stemmed_content = ' '.join(stemmed_content) # join the words into a sentence
  return stemmed_content


In [26]:
twitter_data['stemmed_content'] = twitter_data['text'].apply(stemming)

In [31]:
twitter_data.shape

(1600000, 7)

In [32]:
print(twitter_data['stemmed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: stemmed_content, Length: 1600000, dtype: object


In [33]:
print(twitter_data['target'])

0          0
1          0
2          0
3          0
4          0
          ..
1599995    1
1599996    1
1599997    1
1599998    1
1599999    1
Name: target, Length: 1600000, dtype: int64


In [34]:
#n separating the data and the label
X = twitter_data['stemmed_content'].values
Y = twitter_data['target'].values

In [35]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [36]:
print(Y)

[0 0 0 ... 1 1 1]


Splitting the data to training data and test data


In [37]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

In [39]:
print(X.shape , X_train.shape ,X_test.shape )

(1600000,) (1280000,) (320000,)


In [44]:
print(X_train)

['watch saw iv drink lil wine' 'hatermagazin'
 'even though favourit drink think vodka coke wipe mind time think im gonna find new drink'
 ... 'eager monday afternoon'
 'hope everyon mother great day wait hear guy store tomorrow'
 'love wake folger bad voic deeper']


In [43]:
print(X_test)

['mmangen fine much time chat twitter hubbi back summer amp tend domin free time'
 'ah may show w ruth kim amp geoffrey sanhueza'
 'ishatara mayb bay area thang dammit' ...
 'destini nevertheless hooray member wonder safe trip' 'feel well'
 'supersandro thank']


In [45]:
#Converting the textual data to numerical data

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [46]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9453092 stored elements and shape (1280000, 461488)>
  Coords	Values
  (0, 436713)	0.27259876264838384
  (0, 354543)	0.3588091611460021
  (0, 185193)	0.5277679060576009
  (0, 109306)	0.3753708587402299
  (0, 235045)	0.41996827700291095
  (0, 443066)	0.4484755317023172
  (1, 160636)	1.0
  (2, 109306)	0.4591176413728317
  (2, 124484)	0.1892155960801415
  (2, 407301)	0.18709338684973031
  (2, 129411)	0.29074192727957143
  (2, 406399)	0.32105459490875526
  (2, 433560)	0.3296595898028565
  (2, 77929)	0.31284080750346344
  (2, 443430)	0.3348599670252845
  (2, 266729)	0.24123230668976975
  (2, 409143)	0.15169282335109835
  (2, 178061)	0.1619010109445149
  (2, 150715)	0.18803850583207948
  (2, 132311)	0.2028971570399794
  (2, 288470)	0.16786949597862733
  (3, 406399)	0.29029991238662284
  (3, 158711)	0.4456939372299574
  (3, 151770)	0.278559647704793
  (3, 56476)	0.5200465453608686
  :	:
  (1279996, 318303)	0.21254698865277744
  (12

In [48]:
print(X_test.shape)

(320000, 461488)


Training the logistic regression model:


In [49]:
model = LogisticRegression(max_iter=1000)

In [50]:
model.fit(X_train, Y_train)

LogisticRegression(max_iter=1000)

Model Evaluation

    Accuracy Score
    

In [55]:
#accuracy score on the trainiing data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(Y_train ,  X_train_prediction )


In [56]:
print('Accuracy score on the training data : ' , training_data_accuracy)

Accuracy score on the training data :  0.79871953125


In [57]:
#accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(Y_test ,  X_test_prediction )


In [59]:
print('Accuracy score on the test data : ' , test_data_accuracy)

Accuracy score on the test data :  0.77668125


Model accuracy = 77.67 %

SAVING THE TRAINED MODEL

In [60]:
import pickle

In [61]:
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))

Using the saved model for future prediction

In [62]:
# loading the saved model
loaded_model = pickle.load(open('/content/trained_model.sav', 'rb'))

In [68]:
X_new = X_test[3]
print(Y_test[3])

prediction = loaded_model.predict(X_new)
print(prediction)
if (prediction[0]==0):
  print('Negative tweet')
else:
  print('Positive tweet')

0
[0]
Negative tweet


use this to made the model struggle

Honestly, I really wanted to love this new phone because the design is beautiful,
the screen looks amazing, the camera takes incredible pictures, and the battery lasts much longer
than my old phone. However, after using it for a few days, I started noticing so many annoying
problems: the phone gets extremely hot when I use certain apps, some notifications arrive very late,
the fingerprint sensor sometimes refuses to work, and the latest update somehow made everything
slower. Customer support was friendly and polite, which I appreciate, but they basically told me
to restart the phone and wait for another update. I don't completely regret buying it because there
are definitely things I enjoy, but considering how expensive it was, I expected a much smoother
experience. Maybe future updates will fix everything, but right now I'm impressed and disappointed
at the same time, which is probably the most confusing review I've ever written.

In [73]:
# or predict manually
message = input("Enter a tweet: ")

new_data = vectorizer.transform([message])

prediction = loaded_model.predict(new_data)

print(prediction)
if (prediction[0]==0):
  print('Negative tweet')
else:
  print('Positive tweet')

Enter a tweet: Honestly, I really wanted to love this new phone because the design is beautiful, the screen looks amazing, the camera takes incredible pictures, and the battery lasts much longer than my old phone. However, after using it for a few days, I started noticing so many annoying problems: the phone gets extremely hot when I use certain apps, some notifications arrive very late, the fingerprint sensor sometimes refuses to work, and the latest update somehow made everything slower. Customer support was friendly and polite, which I appreciate, but they basically told me to restart the phone and wait for another update. I don't completely regret buying it because there are definitely things I enjoy, but considering how expensive it was, I expected a much smoother experience. Maybe future updates will fix everything, but right now I'm impressed and disappointed at the same time, which is probably the most confusing review I've ever written.
[0]
Negative tweet
